# Supermarket Pricing Dashboard — Initial Analysis

This notebook performs an initial exploratory analysis of the supermarket pricing dataset.

## Goals
- Load and inspect the dataset
- Check data quality and coverage
- Compare prices across chains and categories
- Explore private label vs branded products
- Identify outliers
- Review cost-per-use metrics where available

## 1. Imports and notebook settings

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)

## 2. Load data

In [ ]:
df = pd.read_excel("Dataset markets.xlsx")

# Standardize column names just in case
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

df.head()

## 3. Quick overview

In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes)

print("\nDataset preview:")
display(df.head())

## 4. Data quality checks
### 4.1 Missing values

In [ ]:
missing_summary = df.isna().sum().sort_values(ascending=False)
display(missing_summary)

### 4.2 Duplicates

In [ ]:
duplicates = df.duplicated().sum()
print("Duplicate rows:", duplicates)

### 4.3 Unique values in key columns

In [ ]:
for col in ["chain", "category", "private_label"]:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False))

### 4.4 Coverage by product_id
This check helps identify products that are missing from one or more chains.

In [ ]:
coverage = (
    df.groupby("product_id")["chain"]
    .nunique()
    .sort_values()
    .reset_index(name="n_chains")
)

display(coverage)
display(coverage[coverage["n_chains"] < 5])

## 5. Descriptive analysis
### 5.1 Average normalized price by chain

In [ ]:
avg_price_chain = (
    df.groupby("chain")["price_per_unit"]
    .mean()
    .sort_values()
    .reset_index()
)

display(avg_price_chain.round(3))

### 5.2 Average normalized price by category and chain

In [ ]:
avg_price_category_chain = (
    df.groupby(["category", "chain"])["price_per_unit"]
    .mean()
    .reset_index()
)

pivot_category_chain = avg_price_category_chain.pivot(
    index="category",
    columns="chain",
    values="price_per_unit"
)

display(pivot_category_chain.round(3))

### 5.3 Cheapest chain by category

In [ ]:
cheapest_by_category = (
    avg_price_category_chain.loc[
        avg_price_category_chain.groupby("category")["price_per_unit"].idxmin()
    ]
    .sort_values("category")
)

display(cheapest_by_category.round(3))

### 5.4 Private label vs branded

In [ ]:
pl_vs_brand = (
    df.groupby("private_label")["price_per_unit"]
    .agg(["count", "mean", "median"])
    .reset_index()
)

display(pl_vs_brand.round(4))

pl_vs_brand_cat = (
    df.groupby(["category", "private_label"])["price_per_unit"]
    .mean()
    .reset_index()
)

display(pl_vs_brand_cat.round(4))

## 6. Visualizations

### 6.1 Average normalized price by chain

In [ ]:
avg_price_chain.plot(
    kind="bar",
    x="chain",
    y="price_per_unit",
    legend=False,
    figsize=(8, 4)
)

plt.ylabel("Average price per unit")
plt.title("Average normalized price by chain")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 6.2 Heatmap: average normalized price by category and chain

**Interpretation note:** direct comparisons *between categories* are limited because the dataset mixes `€/kg`, `€/L`, and `€/unit`. This heatmap is most useful for comparing **chains within the same category**.

In [ ]:
plt.figure(figsize=(10, 5))
plt.imshow(pivot_category_chain, aspect="auto")
plt.colorbar(label="Average price per unit")
plt.xticks(range(len(pivot_category_chain.columns)), pivot_category_chain.columns, rotation=45)
plt.yticks(range(len(pivot_category_chain.index)), pivot_category_chain.index)
plt.title("Normalized price by category and chain")
plt.tight_layout()
plt.show()

### 6.3 Bar chart per category
This is usually easier to interpret than a one-row heatmap.

In [ ]:
categories = sorted(df["category"].dropna().unique())

for category in categories:
    subset = df[df["category"] == category]

    avg_prices = (
        subset.groupby("chain")["price_per_unit"]
        .mean()
        .sort_values()
    )

    plt.figure(figsize=(8, 4))
    avg_prices.plot(kind="bar")
    plt.ylabel("Average price per unit")
    plt.title(f"{category.title()} — average normalized price by chain")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 7. Outlier review

### 7.1 Highest normalized prices

In [ ]:
expensive = df.sort_values("price_per_unit", ascending=False)
display(
    expensive[
        ["product_id", "chain", "brand", "package_size", "price_eur", "price_per_unit"]
    ].head(15).round(3)
)

### 7.2 Lowest normalized prices

In [ ]:
cheap = df.sort_values("price_per_unit", ascending=True)
display(
    cheap[
        ["product_id", "chain", "brand", "package_size", "price_eur", "price_per_unit"]
    ].head(15).round(3)
)

## 8. Cost-per-use analysis
Only applicable to products where `cost_per_use` is available.

In [ ]:
cleaning_use = df[df["cost_per_use"].notna()].copy()

display(
    cleaning_use[
        ["product_id", "chain", "brand", "price_eur", "price_per_unit", "uses", "cost_per_use"]
    ].sort_values(["product_id", "chain"]).round(3)
)

In [ ]:
cost_use_summary = (
    df[df["cost_per_use"].notna()]
    .groupby(["product_id", "chain"])["cost_per_use"]
    .mean()
    .reset_index()
)

display(cost_use_summary.sort_values(["product_id", "cost_per_use"]).round(3))

## 9. Notes for the next iteration
Use this section after the second data collection to:
- compare price changes over time
- validate whether outliers persist
- refine branded vs private label comparisons
- build the Streamlit dashboard from the cleaned tables above